# Öğrenci 4 · Satın alma tahmini ve RFM segmentasyonu

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mhmd-Seyed/ecommerce-behavior-analytics/blob/main/notebooks/04_ml_purchase_segmentation.ipynb)

- **Sınıflandırma.** Logistic Regression ve Random Forest ile oturumda satın alma tahmini. Karşılaştırma için AUC-ROC, Accuracy ve satın alma sınıfının F1 skoru.
- **Öznitelik önemi.** Random Forest modelinden tahminde en çok kullanılan özelliklerin puanları.
- **Segmentasyon.** Recency, Frequency ve Monetary değerleriyle K-Means kullanarak dört müşteri kümesi.

Colab’da indirilen dosyayı **Dosya → Not defteri yükle** ile aç. Ortak Drive klasörünün kısayolunu **Drive’ım** altında `ecommerce_project` adıyla ekle, ardından **Çalışma zamanı → Tümünü çalıştır** seçeneğini kullan. Kısayol başka bir konumdaysa `PROJECT_DIR` değerini güncelle.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pyspark==4.0.1"])

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.sql import SparkSession, functions as F

PROJECT_DIR = Path("/content/drive/MyDrive/ecommerce_project" if drive is not None else ".")
SESSIONS_PATH = Path(os.environ.get("ECOMMERCE_SESSIONS_PATH", PROJECT_DIR / "sessions_summary.parquet"))
EVENTS_PATH = Path(os.environ.get("ECOMMERCE_DATA_PATH", PROJECT_DIR / "events_clean.parquet"))
OUTPUT_DIR = Path(os.environ.get("ECOMMERCE_RESULTS_PATH", PROJECT_DIR))

SEED = 42
OBSERVATION_SECONDS = 60
SESSION_KEYS = ["user_id", "user_session"]

for path in (SESSIONS_PATH, EVENTS_PATH):
    if not path.exists():
        raise FileNotFoundError(f"{path} bulunamadı. Drive bağlantısını ve PROJECT_DIR değerini kontrol et.")

spark = (
    SparkSession.builder.appName("Student4_ML_Models")
    .master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"PySpark {spark.version}")

## 1. Oturum hedefini ve ilk dakika özelliklerini hazırla

`sessions_summary.parquet` Öğrenci 2’den gelir. Oturum kimliğini, başlangıç zamanını ve **`has_purchase`** hedefini bu tablodan alırız. `events_clean.parquet` Öğrenci 1’den gelir. Kılavuzdaki altı özelliği bu olaylardan, oturum başlangıcını izleyen ilk 60 saniye için hesaplarız. Özet tablodaki tüm oturuma ait görüntüleme, sepet, fiyat ve süre değerleri tahmin anından sonraki olayları içerebilir.

Model ilk dakika bitiminde devreye girer. Bu süre içinde satın alan veya görüntüleme ve sepet etkileşimi olmayan oturumları çıkarırız. Kalan oturumlarda hedef, aynı oturumun daha sonra satın almaya dönüşmesidir. `purchase_count` ve `total_spent` sınıflandırma girdilerinde yer almaz.

In [ ]:
def require_columns(frame, columns, file_name):
    missing = set(columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{file_name} dosyasında eksik sütunlar: {sorted(missing)}")

sessions_raw = spark.read.parquet(str(SESSIONS_PATH))
events_raw = spark.read.parquet(str(EVENTS_PATH))

require_columns(sessions_raw, [*SESSION_KEYS, "session_start", "has_purchase"], SESSIONS_PATH.name)
require_columns(events_raw, [*SESSION_KEYS, "event_time", "event_type", "product_id", "price"], EVENTS_PATH.name)

sessions = (
    sessions_raw.select(
        "user_id",
        F.trim("user_session").alias("user_session"),
        F.col("session_start").cast("timestamp").alias("session_start"),
        F.col("has_purchase").cast("double").alias("label"),
    )
    .dropna(subset=[*SESSION_KEYS, "session_start"])
    .where((F.length("user_session") > 0) & F.col("label").isin(0.0, 1.0))
)

events = (
    events_raw.select(
        "user_id", "product_id",
        F.trim("user_session").alias("user_session"),
        F.col("event_time").cast("timestamp").alias("event_time"),
        F.lower(F.trim("event_type")).alias("event_type"),
        F.col("price").cast("double").alias("price"),
    )
    .dropna(subset=[*SESSION_KEYS, "event_time"])
    .where(F.length("user_session") > 0)
    .cache()
)
print(f"Ekip verisi · {sessions.count():,} oturum, {events.count():,} olay")

elapsed_us = F.unix_micros("event_time") - F.unix_micros("session_start")
first_minute = events.join(sessions.select(*SESSION_KEYS, "session_start"), SESSION_KEYS).where(
    elapsed_us.between(0, OBSERVATION_SECONDS * 1_000_000)
)
is_view = F.col("event_type") == "view"
is_cart = F.col("event_type") == "cart"
is_behavior = is_view | is_cart
valid_price = (F.col("price") >= 0) & (F.col("price") < float("inf"))

early_features = (
    first_minute.groupBy(*SESSION_KEYS)
    .agg(
        F.count(F.when(is_view, 1)).alias("view_count"),
        F.count(F.when(is_cart, 1)).alias("cart_count"),
        F.countDistinct(F.when(is_behavior, F.col("product_id"))).alias("distinct_products"),
        F.avg(F.when(is_behavior & valid_price, F.col("price"))).alias("avg_product_price"),
        (F.max(F.when(is_behavior, elapsed_us)) / 1_000_000).alias("session_duration_sec"),
        F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("early_purchase"),
    )
    .where((F.col("early_purchase") == 0) & (F.col("view_count") + F.col("cart_count") > 0))
)

feature_cols = [
    "view_count", "cart_count", "distinct_products",
    "avg_product_price", "session_duration_sec", "session_start_hour",
]
model_data = (
    sessions.join(early_features, SESSION_KEYS)
    .withColumn("session_start_hour", F.hour("session_start"))
    .select(*feature_cols, "label")
    .fillna(0, subset=feature_cols)
    .cache()
)
print(f"Sınıflandırmaya uygun oturum: {model_data.count():,}")

## 2. Eğitim ve test kümelerini oluştur

Veri `seed=42` ile yaklaşık **%80 eğitim / %20 test** olarak ayrılır. `StandardScaler` yalnızca eğitim kümesinde öğrenir. Eğitimde bütün pozitifler tutulur ve hedef olarak her pozitif başına yaklaşık 1,5 negatif örnek seçilir. Test kümesi doğal satın alma oranıyla kalır.

In [ ]:
train_df, test_df = model_data.randomSplit([0.8, 0.2], seed=SEED)
train_df = train_df.cache()
test_df = test_df.cache()

def count_labels(frame):
    counts = {int(row.label): row["count"] for row in frame.groupBy("label").count().collect()}
    if set(counts) != {0, 1}:
        raise ValueError("Bu veri bölümünde her iki satın alma sınıfı da bulunmalı.")
    return counts[0], counts[1]

train_neg, train_pos = count_labels(train_df)
test_neg, test_pos = count_labels(test_df)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(assembler.transform(train_df))
scaled_train = scaler_model.transform(assembler.transform(train_df)).select("features", "label")
scaled_test = scaler_model.transform(assembler.transform(test_df)).select("features", "label").cache()

negative_fraction = min(1.5 * train_pos / train_neg, 1.0)
train_balanced = scaled_train.where(F.col("label") == 1.0).unionByName(
    scaled_train.where(F.col("label") == 0.0).sample(
        withReplacement=False, fraction=negative_fraction, seed=SEED,
    )
).cache()
balanced_neg, balanced_pos = count_labels(train_balanced)

print(f"Eğitim (önce)          · 0: {train_neg:,}, 1: {train_pos:,}")
print(f"Eğitim (denge sonrası) · 0: {balanced_neg:,}, 1: {balanced_pos:,}")
print(f"Test                   · 0: {test_neg:,}, 1: {test_pos:,}")
print(f"Test satın alma oranı: {test_pos / (test_neg + test_pos):.2%}")

## 3. İki modeli eğit ve aynı test setinde değerlendir

Lojistik regresyon ve 30 ağaçlı Random Forest aynı dengelenmiş eğitim kümesini kullanır. Her iki modelin AUC-ROC, Accuracy ve pozitif sınıf F1 skoru aynı test kümesinde ölçülür. Dengeleme sınıf oranını değiştirdiği için model skorlarını kalibrasyon yapılmadan gerçek satın alma yüzdesi olarak yorumlamayız.

In [ ]:
lr_model = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20).fit(train_balanced)
rf_model = RandomForestClassifier(
    featuresCol="features", labelCol="label", numTrees=30, seed=SEED,
).fit(train_balanced)

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC",
)
class_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricLabel=1.0,
)

model_names = [("Logistic Regression", lr_model), ("Random Forest", rf_model)]
metric_rows = []
for name, classifier in model_names:
    predictions = classifier.transform(scaled_test).select("label", "rawPrediction", "prediction").cache()
    metric_rows.append((
        name,
        roc_evaluator.evaluate(predictions),
        class_evaluator.setMetricName("accuracy").evaluate(predictions),
        class_evaluator.setMetricName("fMeasureByLabel").evaluate(predictions),
    ))
    predictions.unpersist()

metrics_df = spark.createDataFrame(metric_rows, ["model_name", "auc_roc", "accuracy", "f1_score"])
metrics_df.show(truncate=False)

importance_rows = [
    (name, float(score)) for name, score in zip(feature_cols, rf_model.featureImportances)
]
fi_df = spark.createDataFrame(importance_rows, ["feature_name", "importance_score"]).orderBy(
    F.desc("importance_score"), F.asc("feature_name"),
)
fi_df.show(truncate=False)
print(f"Her oturumu satın almasız sayan temel yaklaşımın doğruluğu: {test_neg / (test_neg + test_pos):.2%}")

for frame in (train_balanced, scaled_test, train_df, test_df, model_data):
    frame.unpersist()

## 4. RFM ile dört müşteri kümesi oluştur

- **Recency** veri setindeki son tarihle müşterinin son ziyareti arasındaki gün sayısıdır.
- **Frequency** müşterinin farklı oturum sayısıdır.
- **Monetary** satın alma olaylarındaki fiyatların toplamıdır.

RFM vektörü standartlaştırılır ve `KMeans(k=4, seed=42)` eğitilir. Küme numaraları Spark’ın atadığı kimliklerdir. Müşteri profillerini belirlemek için her kümenin ortalama değerlerine bakılır.

In [ ]:
reference_date = events.agg(F.max(F.to_date("event_time")).alias("reference_date")).first().reference_date
if reference_date is None:
    raise ValueError("RFM için geçerli olay tarihi bulunamadı.")

rfm = events.groupBy("user_id").agg(
    F.datediff(F.lit(reference_date), F.max(F.to_date("event_time"))).alias("recency_days"),
    F.countDistinct("user_session").alias("frequency_sessions"),
    F.round(F.sum(F.when((F.col("event_type") == "purchase") & valid_price, F.col("price")).otherwise(0)), 2)
    .alias("monetary_spent"),
).cache()

rfm_assembler = VectorAssembler(
    inputCols=["recency_days", "frequency_sessions", "monetary_spent"], outputCol="rfm_raw",
)
rfm_scaler = StandardScaler(inputCol="rfm_raw", outputCol="features", withStd=True, withMean=True)
kmeans = KMeans(featuresCol="features", predictionCol="cluster_id", k=4, seed=SEED)
rfm_model = Pipeline(stages=[rfm_assembler, rfm_scaler, kmeans]).fit(rfm)

rfm_summary_df = (
    rfm_model.transform(rfm)
    .groupBy("cluster_id")
    .agg(
        F.count("*").alias("user_count"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency_days"),
        F.round(F.avg("frequency_sessions"), 2).alias("avg_frequency"),
        F.round(F.avg("monetary_spent"), 2).alias("avg_monetary_spent"),
    )
    .orderBy("cluster_id")
    .cache()
)
print(f"RFM referans tarihi (UTC) · {reference_date}")
rfm_summary_df.show(truncate=False)

## 5. Öğrenci 5 için üç CSV dosyası üret

Colab’da dosyalar ortak `ecommerce_project` klasörüne yazılır. Çalışmayı GitHub’a kaydetmeden önce Colab menüsünden tüm hücre çıktılarını temizle ve **Dosya → GitHub’da bir kopyasını kaydet** adımıyla `notebooks/04_ml_purchase_segmentation.ipynb` dosyasını güncelle.

In [ ]:
import csv

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "ml_model_metrics.csv": metrics_df,
    "feature_importance.csv": fi_df,
    "rfm_clusters.csv": rfm_summary_df,
}
for name, summary in outputs.items():
    path = OUTPUT_DIR / name
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(summary.columns)
        writer.writerows(summary.collect())
    print(f"Kaydedildi · {name}")

model_directory = os.environ.get("ECOMMERCE_MODEL_DIR")
if model_directory:
    paths = Path(model_directory)
    paths.mkdir(parents=True, exist_ok=True)
    PipelineModel(stages=[assembler, scaler_model, lr_model]).write().overwrite().save(
        str(paths / "logistic_regression")
    )
    PipelineModel(stages=[assembler, scaler_model, rf_model]).write().overwrite().save(
        str(paths / "random_forest")
    )
    rfm_model.write().overwrite().save(str(paths / "rfm_kmeans"))

for frame in (rfm_summary_df, rfm, events):
    frame.unpersist()

### Sunumda kullan

İki modelin AUC-ROC ve satın alma sınıfı F1 skorlarını aynı test kümesinde karşılaştır. Accuracy değerini satın alma oranıyla birlikte yorumla. Random Forest tablosundaki ilk iki özelliği belirt. Önem puanı modelin özelliği ayırım yapmak için ne kadar kullandığını gösterir ve nedensellik sonucu vermez. RFM kümelerine ortalama ziyaret zamanı, ziyaret sıklığı ve harcama değerlerine bakarak isim ver.